<a href="https://colab.research.google.com/github/NikitaIvagin/ml-portfolio/blob/main/nlp/telegram_qa_bot/Telegram_QA_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai
!pip install tiktoken
!pip install wikipedia
import os
import ast
import getpass
import tiktoken
import wikipedia
import pandas as pd
from openai import OpenAI
from google.colab import drive
from scipy import spatial  # вычисляет сходство векторов

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/TES.csv')

# Конвертируем наши эмбединги из строк в списки
df['embedding'] = df['embedding'].apply(ast.literal_eval)
GPT_MODEL = "gpt-3.5-turbo"

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=e067290b685ebe6c22987867c5522573ba6371c801a1650558e2377e658233a6
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Наш датафрейм состоит из 2 колонок text и embedding
df.head()

,text,embedding
0,The Elder Scrolls Adventures: Redguard\n\n{{sh...,"[0.012038983404636383, -0.04322366416454315, -..."
1,The Elder Scrolls Adventures: Redguard\n\n==Ga...,"[0.006428788881748915, -0.03549525886774063, -..."
2,The Elder Scrolls Adventures: Redguard\n\n==Pl...,"[0.008901168592274189, -0.03211347758769989, -..."
3,The Elder Scrolls Adventures: Redguard\n\n==De...,"[0.01405079010874033, -0.035591740161180496, -..."
4,The Elder Scrolls Adventures: Redguard\n\n==Re...,"[0.018037505447864532, -0.040557861328125, -0...."


In [ ]:
# Помещаем OpenAI ключ в переменную окружения
os.environ["OPENAI_API_KEY"] = getpass.getpass("Введите OpenAI API Key:")

openai = OpenAI(api_key = os.environ.get("OPENAI_API_KEY"),)

Введите OpenAI API Key:··········


In [ ]:
EMBEDDING_MODEL = "text-embedding-ada-002"

# Функция поиска
def strings_ranked_by_relatedness(
    query: str, # пользовательский запрос
    df: pd.DataFrame, # DataFrame со столбцами text и embedding (база знаний)
    relatedness_fn=lambda x, y: 1 - spatial.distance.cosine(x, y), # функция схожести, косинусное расстояние
    top_n: int = 100 # выбор лучших n-результатов
) -> tuple[list[str], list[float]]: # Функция возвращает кортеж двух списков, первый содержит строки, второй - числа с плавающей запятой
    """Возвращает строки и схожести, отсортированные от большего к меньшему"""

    # Отправляем в OpenAI API пользовательский запрос для токенизации
    query_embedding_response = openai.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query,
    )

    # Получен токенизированный пользовательский запрос
    query_embedding = query_embedding_response.data[0].embedding

    # Сравниваем пользовательский запрос с каждой токенизированной строкой DataFrame
    strings_and_relatednesses = [
        (row["text"], relatedness_fn(query_embedding, row["embedding"]))
        for i, row in df.iterrows()
    ]

    # Сортируем по убыванию схожести полученный список
    strings_and_relatednesses.sort(key=lambda x: x[1], reverse=True)

    # Преобразовываем наш список в кортеж из списков
    strings, relatednesses = zip(*strings_and_relatednesses)

    # Возвращаем n лучших результатов
    return strings[:top_n], relatednesses[:top_n]

In [ ]:
def num_tokens(text: str, model: str = GPT_MODEL) -> int:
    """Возвращает число токенов в строке для заданной модели"""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

# Функция формирования запроса к chatGPT по пользовательскому вопросу и базе знаний
def query_message(
    query: str, # пользовательский запрос
    df: pd.DataFrame, # DataFrame со столбцами text и embedding (база знаний)
    model: str, # модель
    token_budget: int # ограничение на число отсылаемых токенов в модель
) -> str:
    """Возвращает сообщение для GPT с соответствующими исходными текстами, извлеченными из фрейма данных (базы знаний)."""
    strings, relatednesses = strings_ranked_by_relatedness(query, df) # функция ранжирования базы знаний по пользовательскому запросу
    # Шаблон инструкции для chatGPT
    message = 'Use the below articles on The Elder Scrolls games to answer the subsequent question. If the answer cannot be found in the articles, write "I could not find an answer."'
    # Шаблон для вопроса
    question = f"\n\nQuestion: {query}"

    # Добавляем к сообщению для chatGPT релевантные строки из базы знаний, пока не выйдем за допустимое число токенов
    for string in strings:
        next_article = f'\n\nWikipedia article section:\n"""\n{string}\n"""'
        if (num_tokens(message + next_article + question, model=model) > token_budget):
            break
        else:
            message += next_article
    return message + question


def ask(
    query: str, # пользовательский запрос
    df: pd.DataFrame = df, # DataFrame со столбцами text и embedding (база знаний)
    model: str = GPT_MODEL, # модель
    token_budget: int = 4096 - 500, # ограничение на число отсылаемых токенов в модель
) -> str:
    """Отвечает на вопрос, используя GPT и базу знаний."""
    # Формируем сообщение к chatGPT (функция выше)
    message = query_message(query, df, model=model, token_budget=token_budget)
    messages = [
        {"role": "system", "content": "You answer questions about the Elder Scrolls games."},
        {"role": "user", "content": message},
    ]
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0 # гиперпараметр степени случайности при генерации текста. Влияет на то, как модель выбирает следующее слово в последовательности.
    )
    response_message = response.choices[0].message.content
    return response_message

In [ ]:
!pip install aiogram
import asyncio
import logging
from aiogram import Bot, Dispatcher, types
from aiogram.filters.command import Command
from aiogram.utils.keyboard import InlineKeyboardBuilder, ReplyKeyboardBuilder
from aiogram import F

!pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 698.2/698.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.4 MB/s eta 0:00:00
  Attempting uninstall: aiohttp
    Found existing installation: aiohttp 3.13.2
    Uninstalling aiohttp-3.13.2:
      Successfully uninstalled aiohttp-3.13.2


In [ ]:
# Включаем логирование, чтобы не пропустить важные сообщения
logging.basicConfig(level=logging.INFO)

os.environ["BOT_TOKEN"] = getpass.getpass("Введите Bot token:")

# Объект бота
bot = Bot(token=os.environ.get("BOT_TOKEN"))
# Диспетчер
dp = Dispatcher()

Введите Bot token:··········


In [ ]:
# Хэндлер на команду /start
@dp.message(Command("start"))
async def cmd_start(message: types.Message):
    builder = ReplyKeyboardBuilder()
    builder.add(types.KeyboardButton(text="/start"))
    builder.add(types.KeyboardButton(text="/help"))
    await message.answer("Добро пожаловать в чат-бот по серии игр The Elder Scrolls!", reply_markup=builder.as_markup(resize_keyboard=True))


# Хэндлер на команду /help
@dp.message(Command("help"))
async def cmd_start(message: types.Message):
    await message.answer(f"""Тематика данного бота - серия игр The Elder Scrolls.
    - Количество записей в базе - {df.shape[0]}.
    - Запросы и ответы идут на английском языке.
    - Пример запроса: 'Who is the main villain of The Elder Scrolls V: Skyrim?'""")


# Этот хэндлер будет обрабатывать все команды, кроме /start и /help
@dp.message(F.text, ~Command("start"), ~Command("help"))
async def answer_the_question(message: types.Message):
    await message.answer(ask(message.text))


In [ ]:
# Запуск процесса поллинга новых апдейтов
async def main():

    await dp.start_polling(bot)

if __name__ == "__main__":
    asyncio.run(main())